# Extracting the Time-Frequency Features

In [ ]:
import os
import sys
# Add the parent directory to the Python path to allow module imports
sys.path.append(os.path.dirname(os.getcwd()))

import json
import numpy as np
import pandas as pd
import shutil
import soundfile as sf
from pathlib import Path
from tqdm.auto import tqdm

from modules.datasets import ICBHIAudioDataset, KAUHAudioDataset
from modules.lungsound import LungSoundAudio
from modules.transforms import *

In [ ]:
DATA_PATH = Path(os.path.join(os.path.dirname(os.getcwd()), "data"))
INTERIM_DATA_FOLDER = DATA_PATH / "interim"
PREPROCESSED_DATA_FOLDER = DATA_PATH / "preprocessed"

if not os.path.exists(INTERIM_DATA_FOLDER):
    raise FileNotFoundError(f"Interim data folder not found at {INTERIM_DATA_FOLDER}. Please run the interim preprocessing step first.")

if not os.path.exists(PREPROCESSED_DATA_FOLDER):
    os.makedirs(PREPROCESSED_DATA_FOLDER)
    print(f"Created preprocessed data folder at {PREPROCESSED_DATA_FOLDER}.")
else:
    if len(os.listdir(PREPROCESSED_DATA_FOLDER)) > 0:
        print(f"[WARNING] Preprocessed data folder already exist and is not empty ({PREPROCESSED_DATA_FOLDER}). Consider deleting it to run the preprocessing step again.")

## Preprocessed

In [ ]:
def save_features_as_npz(features: np.ndarray, sr: int, path: str):
    """
    Saves the extracted features and sampling rate to a .npz file.
    Args:
        features (np.ndarray): The extracted features to be saved.
        sr (int): The sampling rate associated with the features.
        path (str): The path where the features will be saved.
    """
    np.savez(path, features=features, sr=sr)

def preprocess_features(original_data_path: Path, preprocessed_data_path: Path, save_as: str = "npz"):
    """
    Preprocesses the original data and saves the preprocessed data to the specified location using multiple feature extractors.
    Args:
        original_data_path (Path): Path to the original raw data.
        preprocessed_data_path (Path): Path where the preprocessed data will be saved.
        save_as (str): Format to save the preprocessed data ('npy', 'npz', or 'png').
    """

    feature_extractors = {
        "MagSTFT": MagSTFT(),
        "ImagSTFT": ImagSTFT(),
        "RealSTFT": RealSTFT(),
        "Phase": Phase(),
        "MelSpectrogram": MelSpectrogram(n_mels=128),
        "MFCC": MFCC(n_mfcc=128),
        "MFCCDelta": MFCCDelta(n_mfcc=128),
        "Chroma": Chroma(n_chroma=128),
    }

    for extractor_name, extractor in feature_extractors.items():
        for dataset in sorted(original_data_path.iterdir()):
            if not dataset.is_dir():
                continue
            # Grab all files in the dataset, .wav or not, including subdirectories
            all_files = sorted(dataset.rglob("**/*.*"))
            dataset_name = dataset.name

            # Process each file in the dataset
            for file in tqdm(all_files, desc=f"Preprocessing {dataset_name} with {extractor_name}"):
                relative_path = file.parent.relative_to(original_data_path)
                if file.suffix.lower() == ".wav":
                    # Load the audio file using the LungSound class
                    audio = LungSoundAudio(file)
                    # Extract features using the provided feature extractor
                    features = extractor(audio)

                    # Construct the new file path for the preprocessed features
                    diagnosis_name = file.parent.name
                    new_file_name = f"{file.stem}.{save_as}"
                    preprocessed_file_path = preprocessed_data_path / dataset_name / extractor_name / diagnosis_name / new_file_name
                    preprocessed_file_path.parent.mkdir(parents=True, exist_ok=True)

                    # Save the preprocessed audio to the new location
                    if save_as == "npz":
                        save_features_as_npz(features.features, features.sr, preprocessed_file_path)
                    else:
                        raise ValueError(f"Unsupported save format: {save_as}")

                elif file.name == "metadata.csv":
                    # If it's a metadata.csv file, we will copy it to the new location and update the file paths
                    df = pd.read_csv(file)
                    df["FilePath"] = df["FilePath"].apply(lambda x: f"{Path(x).stem}.{save_as}")
                    df.rename(columns={"FilePath": "FileName"}, inplace=True)
                    new_file_path = preprocessed_data_path / relative_path / file.name
                    new_file_path.parent.mkdir(parents=True, exist_ok=True)
                    df.to_csv(new_file_path, index=False)
                else:
                    # If it's not an audio file or metadata.csv, simply copy it to the new location
                    new_file_path = preprocessed_data_path / relative_path / file.name
                    new_file_path.parent.mkdir(parents=True, exist_ok=True)
                    shutil.copy(file, new_file_path)
        
            # Save the preprocessing parameters to a json file
            preprocessing_path = preprocessed_data_path / dataset_name / extractor_name / "features_preprocessing.json"
            feature_extractor = {
                extractor.__class__.__name__: {
                    "params": vars(extractor),
                    "plot_params": extractor.plot_params
                }
            }
            preprocessing = {
                "feature_extractor": feature_extractor,
            }
            with open(preprocessing_path, "w") as f:
                json.dump(preprocessing, f, indent=4, default=str)

    print("-------------------------------------------------------")
    print(f"Preprocessing for {original_data_path.name} completed.")
    print(f"Data saved to {os.path.relpath(preprocessed_data_path, start=os.getcwd())}")

In [ ]:
preprocess_features(INTERIM_DATA_FOLDER, PREPROCESSED_DATA_FOLDER, save_as="npz")